# Two-Step Task

## 1. Load the data

In [113]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

data = pd.read_csv("./reduced_two_step_mixture_data.csv")
data.head()

,trial,controller,choice_stage1,state_stage2,transition,reward,reward_prob
0,1,model-free,0,0,common,0,0.634224
1,2,model-based,0,0,common,1,0.658748
2,3,model-based,0,0,common,1,0.636525
3,4,model-free,0,0,common,0,0.618589
4,5,model-free,0,0,common,0,0.625533


## 2. Model Free Approach

In [114]:
class ModelFreeModel:
    def __init__(self, alpha=0.5):
        self.alpha = alpha
        self.v_table = {}

    def get_value(self, state):
        return self.v_table.get(state, 0.5)

    def update(self, state, reward):
        current_value = self.get_value(state)
        delta_v = self.alpha * (reward - current_value)
        self.v_table[state] = current_value + delta_v

    def get_p(self, state):
        state0_value = self.get_value(0)
        state1_value = self.get_value(1)
        if state0_value == 0 and state1_value == 0:
            p = 0
        else:
            p = state0_value / (state0_value + state1_value)
        p = p if state == 0 else 1 - p
        return p

In [115]:
p_common_reward = []
p_common_unreward = []
p_rare_reward = []
p_rare_unreward = []

model_free_model = ModelFreeModel()
for i, row in data.iterrows():
    state = row["choice_stage1"]
    reward = row["reward"]
    transition = row["transition"]
    model_free_model.update(state, reward)

    p = model_free_model.get_p(state)
    if transition == "common" and reward == 1:
        p_common_reward.append(p)
    elif transition == "common" and reward == 0:
        p_common_unreward.append(p)
    elif transition == "rare" and reward == 1:
        p_rare_reward.append(p)
    elif transition == "rare" and reward == 0:
        p_rare_unreward.append(p)

p_common_reward = sum(p_common_reward) / len(p_common_reward)
p_common_unreward = sum(p_common_unreward) / len(p_common_unreward)
p_rare_reward = sum(p_rare_reward) / len(p_rare_reward)
p_rare_unreward = sum(p_rare_unreward) / len(p_rare_unreward)

In [116]:
fig = go.Figure()
fig.add_trace(go.Bar(x=["reward", "unreward"], y=[p_common_reward, p_common_unreward], name="common"))
fig.add_trace(go.Bar(x=["reward", "unreward"], y=[p_rare_reward, p_rare_unreward], name="rare"))
fig.update_layout(barmode="group", title="Model-Free Approach", width=1000, height=600)
fig.show()

## 3. Model-Based Approach

In [117]:
class ModelBasedModel:
    def __init__(self, alpha=0.5):
        self.alpha = alpha
        self.p_common = 0.8
        self.action_values = {}
        self.state_values = {}

    def get_action_value(self, action):
        return self.action_values.get(action, 0.5)

    def get_state_value(self, state):
        return self.state_values.get(state, 0.5)

    def update(self, state, reward):
        state_value = self.get_state_value(state)
        delta_state_value = self.alpha * (reward - state_value)
        self.state_values[state] = state_value + delta_state_value

        self.action_values[0] = self.p_common * self.get_state_value(0) + (1 - self.p_common) * self.get_state_value(1)
        self.action_values[1] = (1 - self.p_common) * self.get_state_value(0) + self.p_common * self.get_state_value(1)

    def get_p(self, state):
        action0_value = self.get_action_value(0)
        action1_value = self.get_action_value(1)
        if action0_value == 0 and action1_value == 0:
            p = 0
        else:
            p = action0_value / (action0_value + action1_value)
        p = p if state == 0 else 1 - p
        return p

In [118]:
p_common_reward = []
p_common_unreward = []
p_rare_reward = []
p_rare_unreward = []

model_based_model = ModelBasedModel()
for i, row in data.iterrows():
    state = row["choice_stage1"]
    next_state = row["state_stage2"]
    reward = row["reward"]
    transition = row["transition"]
    model_based_model.update(next_state, reward)

    p = model_based_model.get_p(state)
    if transition == "common" and reward == 1:
        p_common_reward.append(p)
    elif transition == "common" and reward == 0:
        p_common_unreward.append(p)
    elif transition == "rare" and reward == 1:
        p_rare_reward.append(p)
    elif transition == "rare" and reward == 0:
        p_rare_unreward.append(p)

p_common_reward = sum(p_common_reward) / len(p_common_reward)
p_common_unreward = sum(p_common_unreward) / len(p_common_unreward)
p_rare_reward = sum(p_rare_reward) / len(p_rare_reward)
p_rare_unreward = sum(p_rare_unreward) / len(p_rare_unreward)

In [119]:
fig = go.Figure()
fig.add_trace(go.Bar(x=["reward", "unreward"], y=[p_common_reward, p_common_unreward], name="common"))
fig.add_trace(go.Bar(x=["reward", "unreward"], y=[p_rare_reward, p_rare_unreward], name="rare"))
fig.update_layout(barmode="group", title="Model-Based Approach", width=1000, height=600)
fig.show()

## 4. Mixed Model

In [120]:
class MixedModel:
    def __init__(self, alpha=0.5, eta=0.0025):
        self.model_free = ModelFreeModel(alpha)
        self.model_based = ModelBasedModel(alpha)
        self.p = 0.5
        self.eta = eta
        self.last_choice = 0 if (np.random.random() < 0.5) else 1

    def update(self, state_free, state_based, reward):
        self.model_free.update(state_free, reward)
        self.model_based.update(state_based, reward)
        self.last_choice = state_free

    def forward(self, state):
        p_free = self.model_free.get_p(state)
        p_based = self.model_based.get_p(state)
        p = self.p * p_free + (1 - self.p) * p_based
        return np.clip(p, 0.001, 0.999)

    def backward(self, state):
        p = self.forward(state)
        p_free = self.model_free.get_p(state)
        p_based = self.model_based.get_p(state)
        # loss = -label * np.log(p) - (1 - label) * np.log(1 - p)
        label = float(state == self.last_choice)
        grad_p = -label / p + (1 - label) / (1 - p)
        grad_self_p = p_free - p_based
        self.p -= self.eta * grad_p * grad_self_p

In [121]:
p_list = [0.5]
p_common_reward = []
p_common_unreward = []
p_rare_reward = []
p_rare_unreward = []

model = MixedModel()
for i, row in data.iterrows():
    state = row["choice_stage1"]
    next_state = row["state_stage2"]
    reward = row["reward"]
    transition = row["transition"]

    model.update(state, next_state, reward)
    model.backward(state)
    p = model.forward(state)
    p_list.append(model.p)

    if transition == "common" and reward == 1:
        p_common_reward.append(p)
    elif transition == "common" and reward == 0:
        p_common_unreward.append(p)
    elif transition == "rare" and reward == 1:
        p_rare_reward.append(p)
    elif transition == "rare" and reward == 0:
        p_rare_unreward.append(p)

p_common_reward = sum(p_common_reward) / len(p_common_reward)
p_common_unreward = sum(p_common_unreward) / len(p_common_unreward)
p_rare_reward = sum(p_rare_reward) / len(p_rare_reward)
p_rare_unreward = sum(p_rare_unreward) / len(p_rare_unreward)

In [122]:
fig = go.Figure()
fig.add_trace(go.Bar(x=["reward", "unreward"], y=[p_common_reward, p_common_unreward], name="common"))
fig.add_trace(go.Bar(x=["reward", "unreward"], y=[p_rare_reward, p_rare_unreward], name="rare"))
fig.update_layout(barmode="group", title="Model-Based Approach", width=1000, height=600)
fig.show()

In [123]:
fig = go.Figure(data=[
    go.Scatter(x=[i for i in range(len(p_list))], y=p_list, name="p")
])
fig.update_layout(title="Mixed Model", width=1000, height=600)
fig.show()

In [124]:
len(data[data["controller"] == "model-free"]) / len(data)

0.38